In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    to_date,
    monotonically_increasing_id
)

import uuid

# =========================================================
# WIDGETS
# =========================================================

dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")

dbutils.widgets.text("partition_column", "")
dbutils.widgets.text("lower_bound", "")
dbutils.widgets.text("upper_bound", "")
dbutils.widgets.text("num_partitions", "")

dbutils.widgets.text("jdbc_url", "")
dbutils.widgets.text("user", "")
dbutils.widgets.text("password", "")
dbutils.widgets.text("driver", "")
dbutils.widgets.text("source_name", "")

# =========================================================
# READ WIDGETS (NO TYPE CASTING FOR BOUNDS)
# =========================================================

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")

partition_column = dbutils.widgets.get("partition_column")

lower_bound = dbutils.widgets.get("lower_bound")
upper_bound = dbutils.widgets.get("upper_bound")

num_partitions_raw = dbutils.widgets.get("num_partitions")

num_partitions = (
    int(float(num_partitions_raw))
    if num_partitions_raw and num_partitions_raw.strip() != ""
    else None
)

jdbc_url = dbutils.widgets.get("jdbc_url")
user = dbutils.widgets.get("user")
password = dbutils.widgets.get("password")
driver = dbutils.widgets.get("driver")
source_name = dbutils.widgets.get("source_name")

# =========================================================
# JDBC READ (SAFE FOR BOTH TIMESTAMP + NUMERIC)
# =========================================================

reader = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", source_table) \
    .option("user", user) \
    .option("password", password) \
    .option("driver", driver)

# Apply partitioning only if provided
if partition_column and lower_bound and upper_bound and num_partitions:

    reader = reader \
        .option("partitionColumn", partition_column) \
        .option("lowerBound", lower_bound) \
        .option("upperBound", upper_bound) \
        .option("numPartitions", num_partitions)

df = reader.load()

# =========================================================
# METADATA GENERATION
# =========================================================

pipeline_run_id = str(uuid.uuid4())

df = (
    df.withColumn("migrated_at", current_timestamp())
      .withColumn("ingestion_date", to_date(current_timestamp()))
      .withColumn("batch_id", lit(f"{lower_bound}_{upper_bound}"))
      .withColumn("source_table", lit(source_table))
      .withColumn("source_system", lit(source_name))
      .withColumn("load_start_id", lit(lower_bound))
      .withColumn("load_end_id", lit(upper_bound))
      .withColumn("pipeline_run_id", lit(pipeline_run_id))
      .withColumn("uniq_id", monotonically_increasing_id())
)

# =========================================================
# WRITE TO DELTA
# =========================================================

df.write.format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print("JDBC PARTITION LOAD SUCCESS")

In [0]:
# from pyspark.sql.functions import (
#     current_timestamp,
#     lit,
#     sha2,
#     concat_ws,
#     to_date,
#     monotonically_increasing_id
# )

# import uuid
# import math

# dbutils.widgets.text("source_table", "")
# dbutils.widgets.text("target_table", "")

# dbutils.widgets.text("partition_column", "")
# dbutils.widgets.text("lower_bound", "")
# dbutils.widgets.text("upper_bound", "")
# dbutils.widgets.text("num_partitions", "")
# dbutils.widgets.text("jdbc_url", "")
# dbutils.widgets.text("user", "")
# dbutils.widgets.text("password", "")
# dbutils.widgets.text("driver", "")
# dbutils.widgets.text("source_name", "")

# source_table = dbutils.widgets.get("source_table")
# target_table = dbutils.widgets.get("target_table")

# partition_column = dbutils.widgets.get("partition_column")

# lower_bound = int(float(dbutils.widgets.get("lower_bound")))
# upper_bound = int(float(dbutils.widgets.get("upper_bound")))
# num_partitions = int(float(dbutils.widgets.get("num_partitions")))

# jdbc_url=dbutils.widgets.get("jdbc_url")
# user=dbutils.widgets.get("user")
# password=dbutils.widgets.get("password")
# driver=dbutils.widgets.get("driver")
# source_name = dbutils.widgets.get("source_name")

# df = spark.read.format("jdbc") \
#     .option("url", jdbc_url) \
#     .option("dbtable", source_table) \
#     .option("user", user) \
#     .option("password", password) \
#     .option("driver", driver) \
#     .option("partitionColumn", partition_column) \
#     .option("lowerBound", lower_bound) \
#     .option("upperBound", upper_bound) \
#     .option("numPartitions", num_partitions) \
#     .load()

# # =========================================================
# # METADATA GENERATION
# # =========================================================

# batch_id = f"{lower_bound}_{upper_bound}"

# pipeline_run_id = str(uuid.uuid4())

# # Keep only original business columns for hash generation
# business_columns = df.columns

# df = (
#     df.withColumn("migrated_at", current_timestamp())
#       .withColumn("ingestion_date", to_date(current_timestamp()))
#       .withColumn("batch_id", lit(batch_id))
#       .withColumn("source_table", lit(source_table))
#       .withColumn("source_system", lit(source_name))
#       .withColumn("load_start_id", lit(lower_bound))
#       .withColumn("load_end_id", lit(upper_bound))
#       .withColumn("pipeline_run_id", lit(pipeline_run_id))
#       .withColumn(
#           "uniq_id",
#           monotonically_increasing_id()
#       )
# )

# df.write.format("delta") \
#     .mode("append") \
#     .saveAsTable(target_table)

# print("JDBC PARTITION LOAD SUCCESS")